The goal of this notebook is to investigate and quantify the change in chlorophyll over the lifetimes of cyclones traveling South and anticyclones traveling North respectively.

What is Known:
- Cyclones tend to have enhanced chlorophyll while anticyclones have suppressed chlorophyl.
- The north side of the Gulf Stream contains cooler nutrient-rich waters while south of the Gulf Stream contains warmer and more oligotrophic waters.

Target eddies:
- Cyclones formed north of the Gulf Stream axis and ended south, or was formed within `NEAR_AXIS_KM` (150 km) of the axis and ended south.
- An anticyclone same rule, reversed.

Eddy requirements:
- 50% coverage from CMEMS
- At least 10 valid pixels
- Within each age bin, all pixels within each eddy-day are averaged, and then all eddy-date composites are averaged again within a bin

The CMEMS data is in 8-day composites (not running mean).

In [ ]:
# Count the number of eddies in our time / regional window which have CHL data from CMEMS matched as well.
from pathlib import Path
from typing import cast
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.axes import Axes
from matplotlib.ticker import AutoLocator

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from eddy_tracking.config import load_config

EXPERIMENT = 'gulf_stream_20240305_20260531'
N_AGE_BINS = 5
N_BOOTSTRAP = 2000
RANDOM_SEED = 2026
EXCLUDE_RECORD_EDGE_TRACKS = False
NEAR_AXIS_KM = 150
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_config(EXPERIMENT)
polarity_names = ('cyclone', 'anticyclone')
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
target_labels = {'cyclone': 'Target cyclones', 'anticyclone': 'Target anticyclones'}
identity_columns = ['polarity', 'track_id']

movement = pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet')
chl = pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_table.parquet')
eddy_tracks = movement.merge(
    cast(pd.Series, chl.groupby(identity_columns).size()).rename('n_chl_dates').reset_index(),
    on=identity_columns, how='left',
)
eddy_tracks['n_chl_dates'] = eddy_tracks['n_chl_dates'].fillna(0).astype(int)
physical_start, physical_end = pd.to_datetime(cfg['base']['time']['eddy_date_range'])
eddy_tracks['at_record_edge'] = (
    (eddy_tracks['birth_date'] <= physical_start)
    | (eddy_tracks['death_date'] >= physical_end)
)
target_class = cast(pd.Series, eddy_tracks['polarity']).map(target_classes)
eddy_tracks['crossed_axis'] = eddy_tracks['movement'].eq(target_class)
eddy_tracks['near_axis_birth'] = (
    eddy_tracks['birth_distance_km'].abs().le(NEAR_AXIS_KM)
    & eddy_tracks['death_side'].eq(target_class.str[1])
)
eddy_tracks['is_target'] = eddy_tracks['crossed_axis'] | eddy_tracks['near_axis_birth']
chl = chl.merge(
    eddy_tracks[identity_columns + ['at_record_edge', 'is_target']],
    on=identity_columns, how='left',
)
target_chl = cast(pd.DataFrame, chl.loc[chl['is_target']]).copy()
if EXCLUDE_RECORD_EDGE_TRACKS:
    target_chl = cast(pd.DataFrame, target_chl.loc[~target_chl['at_record_edge']]).copy()

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    'font.size': 8, 'axes.titlesize': 8.5, 'axes.labelsize': 8, 'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 7,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})


classes = ['NN', 'NS', 'SN', 'SS']
count_rows = []
matched_tracks = eddy_tracks.loc[eddy_tracks['n_chl_dates'].gt(0)]
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9), sharey=True, layout='constrained')
for panel, polarity in enumerate(polarity_names):
    ax = cast(Axes, axes[panel])
    tracks = matched_tracks.loc[matched_tracks['polarity'] == polarity]
    total = tracks.groupby('movement').size().reindex(classes, fill_value=0)
    target = tracks.loc[tracks['is_target']].groupby('movement').size().reindex(classes, fill_value=0)
    positions = np.arange(len(classes))
    all_bars = ax.bar(positions - 0.19, total.to_numpy(), width=0.36, color='#c3c8cd', label='All tracks')
    target_bars = ax.bar(positions + 0.19, target.to_numpy(), width=0.36, color=polarity_colors[polarity], label='Target eddies')
    ax.bar_label(all_bars, padding=2, fontsize=6.5, color='#333333')
    ax.bar_label(target_bars, labels=[str(value) if value else '' for value in target.to_numpy()], padding=2, fontsize=6.5, color='#333333')
    ax.xaxis.set_ticks(positions)
    ax.xaxis.set_ticklabels([label[0] + ' → ' + label[1] for label in classes])
    ax.set_title(f'$\\bf{{({"ab"[panel]})}}$ {polarity.capitalize()}s: {len(tracks)} tracks, {int(target.sum())} target eddies', loc='left')
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.tick_params(axis='x', length=0)
    for label in classes:
        count_rows.append({'polarity': polarity, 'class': label, 'tracks': int(total[label]), 'target': int(target[label])})
axes[0].set_ylim(0, np.ceil(int(matched_tracks.groupby(['polarity', 'movement']).size().max()) * 1.12 / 50) * 50)
axes[0].set_ylabel('Tracks with CHL data')
axes[0].legend(loc='upper left', handlelength=1.2, handleheight=0.9)
fig.supxlabel('First side → last side of the daily Gulf Stream axis', fontsize=8)
class_counts = pd.DataFrame(count_rows)
plt.show()
print('Tracks with CHL data and target eddies in each movement class, the counts behind the bars.')
display(class_counts)
target_rules = {
    'crossed the axis': eddy_tracks['crossed_axis'],
    f'born within {NEAR_AXIS_KM} km, no crossing': eddy_tracks['near_axis_birth'] & ~eddy_tracks['crossed_axis'],
    'target eddies': eddy_tracks['is_target'],
}
print(f'Target eddies by the rule that admits them: an axis crossing, or a birth within {NEAR_AXIS_KM} km of the axis without a crossing, and their sum.')
display(pd.DataFrame([
    {'polarity': polarity, 'rule': rule, 'tracks': int((members & eddy_tracks['polarity'].eq(polarity)).sum())}
    for polarity in polarity_names for rule, members in target_rules.items()
]))

In [ ]:
# Show an example of a target cyclone and anticyclone in the region and its start/end path.
# Line breaks occur when PET found no eddy on 1 or more consecutive days. Settings are max 5 consecutive missed days and tracks running for at least 30 days.
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.geoaxes import GeoAxes
from matplotlib.lines import Line2D
from eddy_tracking.preprocess.tracks import load_track_observations

track_observations = load_track_observations(EXPERIMENT)
crossed = cast(pd.DataFrame, eddy_tracks.loc[eddy_tracks['crossed_axis']]).copy()
crossed['lifetime_days'] = (crossed['death_date'] - crossed['birth_date']).dt.days
representatives = crossed.sort_values('lifetime_days', ascending=False).drop_duplicates('polarity').set_index('polarity')

map_fig, map_ax = plt.subplots(figsize=(7.2, 4.6), subplot_kw={'projection': ccrs.PlateCarree()}, layout='constrained')
map_ax = cast(GeoAxes, map_ax)
map_ax.set_extent([*cfg['base']['region']['lon_range'], *cfg['base']['region']['lat_range']], crs=ccrs.PlateCarree())
map_ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#e9e9e9', zorder=0)
map_ax.coastlines(resolution='50m', color='#666666', linewidth=0.5)
grid = map_ax.gridlines(draw_labels=True, linewidth=0.4, color='#bbbbbb', xlocs=range(-80, -55, 5), ylocs=range(30, 45, 2))
grid.top_labels = False
grid.right_labels = False
grid.xlabel_style = {'size': 7}
grid.ylabel_style = {'size': 7}
handles = []
for polarity in polarity_names:
    track_id = int(representatives.loc[polarity, 'track_id'])
    track = cast(pd.DataFrame, track_observations.loc[
        track_observations['polarity'].eq(polarity) & track_observations['track_id'].eq(track_id)
    ]).sort_values('date')
    first, last = track.iloc[0], track.iloc[-1]
    color = polarity_colors[polarity]
    gap_groups = track['date'].diff().dt.days.gt(1).cumsum()
    for _, segment in track.groupby(gap_groups):
        map_ax.plot(segment['center_lon'], segment['center_lat'], color=color, linewidth=1.3, transform=ccrs.PlateCarree())
    map_ax.scatter(first['center_lon'], first['center_lat'], s=30, marker='o', facecolors='white', edgecolors=color, linewidths=1.2, zorder=5, transform=ccrs.PlateCarree())
    map_ax.scatter(last['center_lon'], last['center_lat'], s=36, marker='^', color=color, edgecolors='white', linewidths=0.6, zorder=5, transform=ccrs.PlateCarree())
    handles.append(Line2D([], [], color=color, linewidth=1.3, label=f'{polarity.capitalize()} {track_id}, {first["date"]:%Y-%m-%d} to {last["date"]:%Y-%m-%d}, {len(track)} detections'))
handles.append(Line2D([], [], linestyle='none', marker='o', markerfacecolor='white', markeredgecolor='#333333', markersize=5, label='First detection'))
handles.append(Line2D([], [], linestyle='none', marker='^', color='#333333', markersize=5.5, label='Last detection'))
map_ax.legend(handles=handles, loc='lower right', frameon=True, framealpha=0.95, edgecolor='none')
plt.show()

In [ ]:
# Compare CMEMS CHL in target cyclones/anticyclones across their lifetime.
bin_edges = np.linspace(0, 1, N_AGE_BINS + 1)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
analysis = target_chl.sort_values(identity_columns + ['date']).copy()
analysis['age_bin'] = np.minimum(
    (analysis['age_frac'] * N_AGE_BINS).astype(int), N_AGE_BINS - 1,
)
analysis['initial_chl'] = analysis.groupby(identity_columns)['CHL'].transform('first')
analysis['chl_change'] = analysis['CHL'] - analysis['initial_chl']
eddy_bins = cast(pd.DataFrame, analysis.groupby(identity_columns + ['age_bin']).agg(
    chl_mean=('CHL', 'mean'), chl_change=('chl_change', 'mean'),
    n_composites=('date', 'size'),
)).reset_index()
rng = np.random.default_rng(RANDOM_SEED)
summary_rows = []
for polarity in polarity_names:
    polarity_bins = eddy_bins.loc[eddy_bins['polarity'].eq(polarity)]
    eddy_ids = sorted(polarity_bins['track_id'].unique())
    draws = rng.integers(0, len(eddy_ids), size=(N_BOOTSTRAP, len(eddy_ids)))
    for metric in ('chl_mean', 'chl_change'):
        matrix = polarity_bins.pivot(index='track_id', columns='age_bin', values=metric).reindex(index=eddy_ids, columns=range(N_AGE_BINS)).to_numpy(dtype=float)
        finite = np.isfinite(matrix)
        counts = finite.sum(axis=0)
        means = np.divide(np.nansum(matrix, axis=0), counts, out=np.full(N_AGE_BINS, np.nan), where=counts > 0)
        sampled = matrix[draws]  # (n_eddies, n_bins) -> (n_bootstrap, n_eddies, n_bins)
        sampled_counts = np.isfinite(sampled).sum(axis=1)
        sampled_means = np.divide(
            np.nansum(sampled, axis=1), sampled_counts,
            out=np.full((N_BOOTSTRAP, N_AGE_BINS), np.nan), where=sampled_counts > 0,
        )
        for age_bin in range(N_AGE_BINS):
            bootstrap_values = sampled_means[:, age_bin]  # (n_bootstrap, n_bins) -> (n_bootstrap,)
            bootstrap_values = bootstrap_values[np.isfinite(bootstrap_values)]
            low, high = (np.quantile(bootstrap_values, [0.025, 0.975]) if counts[age_bin] >= 3 else (np.nan, np.nan))
            summary_rows.append({
                'polarity': polarity, 'metric': metric, 'age_bin': age_bin,
                'age_midpoint': bin_centers[age_bin], 'mean': means[age_bin],
                'ci_low': low, 'ci_high': high, 'n_eddies': int(counts[age_bin]),
            })
lifetime_summary = pd.DataFrame(summary_rows)
target_summary = cast(pd.DataFrame, analysis.groupby('polarity').agg(
    eddies=('track_id', 'nunique'), composites=('date', 'size'),
    first_composite=('date', 'min'), last_composite=('date', 'max'),
))
record_edge_counts = analysis.loc[analysis['at_record_edge']].drop_duplicates(identity_columns).groupby('polarity').size()
target_summary['tracks_at_record_edge'] = record_edge_counts.reindex(target_summary.index, fill_value=0)

life_fig, life_axes = plt.subplots(1, 2, figsize=(7.2, 3.1), layout='constrained')
for panel, metric in enumerate(('chl_mean', 'chl_change')):
    ax = cast(Axes, life_axes[panel])
    if metric == 'chl_change':
        ax.axhline(0, color='#999999', linewidth=0.6, zorder=1)
    for polarity in polarity_names:
        result = lifetime_summary.loc[
            lifetime_summary['polarity'].eq(polarity) & lifetime_summary['metric'].eq(metric)
        ].sort_values('age_bin')
        color = polarity_colors[polarity]
        intervals = result.loc[result['ci_low'].notna()]
        ax.errorbar(
            intervals['age_midpoint'], intervals['mean'],
            yerr=[intervals['mean'] - intervals['ci_low'], intervals['ci_high'] - intervals['mean']],
            fmt='none', ecolor=color, capsize=2, elinewidth=0.8, capthick=0.8, zorder=2,
        )
        ax.plot(result['age_midpoint'], result['mean'], '-o', color=color, markersize=4, markeredgecolor='white', markeredgewidth=0.6, label=f'{target_labels[polarity]} (n = {target_summary.loc[polarity, "eddies"]})', zorder=3)
    ax.set_xlabel('Fraction of observed track')
    ax.xaxis.set_ticks(np.linspace(0, 1, 6))
    ax.set_xlim(0, 1)
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    ticks = cast(np.ndarray, AutoLocator().tick_values(*ax.get_ylim()))
    ax.set_ylim(ticks[0], ticks[-1])
life_axes[0].set_title('$\\bf{(a)}$ Interior CHL', loc='left')
life_axes[0].set_ylabel('CHL (mg m$^{-3}$)')
life_axes[1].set_title('$\\bf{(b)}$ Change from first observation', loc='left')
life_axes[1].set_ylabel('$\\Delta$CHL (mg m$^{-3}$)')
life_fig.legend(*life_axes[0].get_legend_handles_labels(), loc='outside lower center', ncol=2)
plt.show()
print('Eddies, composites, composite date range, and tracks at the record edge per polarity.')
display(target_summary)
print('Mean interior CHL per bin with the 95% interval and the number of eddies behind it, the values of panel (a).')
display(lifetime_summary.loc[lifetime_summary['metric'].eq('chl_mean')].drop(columns='metric').round(4))

In [ ]:
import datetime as dt

import xarray as xr
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

from eddy_tracking.preprocess.tracks import PET_EPOCH
from eddy_tracking.packages.py_eddy_tracker.observations.tracking import TrackEddiesObservations

windows = {}
for year in range(physical_start.year, physical_end.year + 1):
    start = dt.date(year, 1, 1)
    while start.year == year:
        end = min(start + dt.timedelta(days=7), dt.date(year, 12, 31))
        windows[pd.Timestamp(start + (end - start) / 2)] = (str(start), str(end))
        start = end + dt.timedelta(days=1)
radius_frames = []
for polarity in polarity_names:
    tracked = TrackEddiesObservations.load_file(str(DATA_DIR / f'silver/eddy_track/{polarity}/{polarity}_tracks.zarr'))
    keep = ~tracked.virtual.astype(bool)
    radius_frames.append(pd.DataFrame({
        'polarity': polarity, 'track_id': tracked.track[keep].astype(int),
        'date': pd.Timestamp(PET_EPOCH) + pd.to_timedelta(tracked.time[keep].astype(int), unit='D'),
        'radius_km': tracked.radius_s[keep] / 1000,
    }))
profiles = pd.merge_asof(
    analysis.sort_values('date'), pd.concat(radius_frames).sort_values('date'),
    on='date', by=identity_columns, direction='nearest',
)
chl_field = xr.open_mfdataset(sorted((DATA_DIR / 'bronze/plankton').glob('plankton_*.nc')), combine='by_coords')['CHL']
lon = chl_field['longitude'].to_numpy()
lat = chl_field['latitude'].to_numpy()
radial_edges = np.linspace(0, 2, 11)
ring_chl = np.full((len(profiles), len(radial_edges) - 1), np.nan)
for date, rows in profiles.groupby('date'):
    composite = chl_field.sel(time=slice(*windows[date])).mean('time').to_numpy()
    for index, center_lon, center_lat, radius_km in zip(rows.index, rows['center_lon'], rows['center_lat'], rows['radius_km']):
        half_lat = 2 * radius_km / 111.0
        lat_idx = np.flatnonzero(np.abs(lat - center_lat) <= half_lat)
        lon_idx = np.flatnonzero(np.abs(lon - center_lon) <= half_lat / np.cos(np.radians(center_lat)))
        lon_grid, lat_grid = np.meshgrid(lon[lon_idx], lat[lat_idx])
        distance = np.hypot((lon_grid - center_lon) * np.cos(np.radians(center_lat)), lat_grid - center_lat) * 111.0 / radius_km
        values = composite[np.ix_(lat_idx, lon_idx)]
        valid = np.isfinite(values) & (distance < 2)
        ring = np.digitize(distance[valid], radial_edges) - 1
        counts = np.bincount(ring, minlength=len(radial_edges) - 1)
        sums = np.bincount(ring, weights=values[valid], minlength=len(radial_edges) - 1)
        ring_chl[index] = np.divide(sums, counts, out=np.full(len(radial_edges) - 1, np.nan), where=counts > 0)
radial = profiles[identity_columns + ['date', 'age_bin']].join(
    pd.DataFrame(ring_chl, columns=range(len(radial_edges) - 1)),
).melt(id_vars=identity_columns + ['date', 'age_bin'], var_name='radial_bin', value_name='CHL')
radial_grid = radial.groupby(identity_columns + ['age_bin', 'radial_bin'])['CHL'].mean().groupby(['polarity', 'age_bin', 'radial_bin']).mean()
chl_norm = Normalize(np.floor(radial_grid.min() / 0.02) * 0.02, np.ceil(radial_grid.max() / 0.02) * 0.02)

radial_fig, radial_axes = plt.subplots(1, 2, figsize=(4.9, 4.0), sharey=True, layout='constrained')
for panel, polarity in enumerate(polarity_names):
    ax = cast(Axes, radial_axes[panel])
    matrix = radial_grid.loc[polarity].unstack('radial_bin').reindex(index=range(N_AGE_BINS), columns=range(len(radial_edges) - 1)).to_numpy().T
    ax.pcolormesh(bin_edges, radial_edges, matrix, cmap='viridis', norm=chl_norm, edgecolors='white', linewidth=0.4)
    ax.axhline(1, color='#222222', linewidth=0.9, linestyle=(0, (4, 2.5)), zorder=3)
    ax.set_aspect('equal')
    ax.set_xlabel('Fraction of observed track')
    ax.xaxis.set_ticks(bin_edges)
    ax.yaxis.set_ticks(radial_edges)
    ax.tick_params(length=2)
    ax.set_title(f'$\\bf{{({"ab"[panel]})}}$ {target_labels[polarity]} (n = {profiles.loc[profiles["polarity"].eq(polarity), "track_id"].nunique()})', loc='left', fontsize=8)
radial_axes[0].text(0.03, 1.03, 'speed contour', fontsize=6.5, color='#222222', va='bottom', ha='left')
radial_axes[0].set_ylabel('Distance from eddy center (speed radii)')
colorbar = radial_fig.colorbar(ScalarMappable(norm=chl_norm, cmap='viridis'), cax=cast(Axes, radial_axes[1]).inset_axes((1.08, 0, 0.06, 1)), label='CHL (mg m$^{-3}$)')
colorbar.ax.tick_params(labelsize=7, length=2)
colorbar.outline.set_linewidth(0.6)
plt.show()
print('Mean CHL in each age bin (rows) and ring (columns, the index of the ring from the center), the values of the color map.')
display(radial_grid.unstack('radial_bin').round(3))